# Model Error Analysis

This notebook loads a trained tracking model and analyzes its predictions and losses for error analysis.

In [2]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # Added missing import
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# Add src to path
import sys
sys.path.append('./src')

from utils import set_seed
from utils.get_data import get_data_loader, get_dataset
from utils.get_model import get_model
from utils import get_loss
from gnn_tracking_prev.metrics.cluster_metrics import tracking_metrics


# Set device
device = torch.device('cuda:5' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda:5


## Configuration and Model Loading

In [3]:
# Load configuration
config_path = "./configs/tracking/tracking_trans_hept.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
for key, value in config.items():
    print(f"{key}: {value}")

# Override config for analysis (you can modify these)
# Specify the checkpoint to load
checkpoint_name = "07_10-23_31_03.0171_trans_hept_42_new_mask_weights_and_decoder_2_hdim_24_R20"  # Change this to your checkpoint folder name
config["resume"] = checkpoint_name

print(f"\nUsing checkpoint: {checkpoint_name}")

# Set seed for reproducibility
set_seed(config['seed'])

Configuration loaded:
device: cuda:4
note: new_mask_weights_and_decoder_2_hdim_24_R5_bcemlp
seed: 42
log_wandb: True
num_threads: 5
resume: 07_10-23_31_03.0171_trans_hept_42_new_mask_weights_and_decoder_2_hdim_24_R20
only_eval: False
model_name: trans_hept
model_kwargs: {'block_size': 350, 'n_hashes': 3, 'num_regions': 20, 'pe_type': 'none', 'num_heads': 4, 'h_dim': 24, 'n_layers': 4, 'num_w_per_dist': 10, 'use_ckpt': True, 'num_queries': 1950, 'num_dec_layers': 2, 'deep_supervision': True, 'reinit_regions': False}
loss_name: set
loss_kwargs: {'loss_ce': 0.4, 'loss_mask': 400, 'loss_dice': 2, 'eos_coef': 1, 'clf_loss': 1.0}
optimizer_name: adam
num_epochs: 250
batch_size: 1
optimizer_kwargs: {'lr': 0.00025}
lr_scheduler_name: step
lr_scheduler_kwargs: {'gamma': 0.5, 'step_size': 100}
data_dir: ../data/
dataset_name: trackml-large

Using checkpoint: 07_10-23_31_03.0171_trans_hept_42_new_mask_weights_and_decoder_2_hdim_24_R20


In [4]:
# Load dataset
dataset_name = config["dataset_name"]
dataset_dir = Path(config["data_dir"]) / "tracking"
dataset = get_dataset(dataset_name, dataset_dir)
loaders = get_data_loader(dataset, dataset.idx_split, batch_size=1)  # Use batch_size=1 for analysis

print(f"Dataset: {dataset_name}")
print(f"Train samples: {len(dataset.idx_split['train'])}")
print(f"Valid samples: {len(dataset.idx_split['valid'])}")
print(f"Test samples: {len(dataset.idx_split['test'])}")

Dataset: trackml-large
Train samples: 1520
Valid samples: 190
Test samples: 190


In [5]:
# Load model
model_name = config["model_name"]
model = get_model(model_name, config["model_kwargs"], dataset)

# Load trained weights
if config.get("resume", False):
    model_path = dataset_dir / "logs" / (config["resume"] + "/best_model.pt")
    print(f"Loading model from: {model_path}")
    checkpoint = torch.load(model_path, map_location="cpu")

    # Handle region reinitialization if needed
    if config["model_kwargs"]["reinit_regions"]:
        current_regions_shape = checkpoint["regions"].shape
        new_num_regions = config["model_kwargs"].get("num_regions")
        new_n_hashes = config["model_kwargs"].get("n_hashes")
        new_num_heads = config["model_kwargs"].get("num_heads")

        model.reinit_regions(new_num_regions, new_n_hashes, new_num_heads)
        print(f"Reinitialized regions: old shape {current_regions_shape}, new num_regions={new_num_regions}")
        checkpoint = {k: v for k, v in checkpoint.items() if k != "regions"}

    model.load_state_dict(checkpoint, strict=False)
    print("Model weights loaded successfully!")
else:
    print("No checkpoint specified, using random weights")

model = model.to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")

Number of parameters: 322370
Loading model from: ../data/tracking/logs/07_10-23_31_03.0171_trans_hept_42_new_mask_weights_and_decoder_2_hdim_24_R20/best_model.pt
Model weights loaded successfully!
Model: trans_hept
Number of parameters: 323846


In [6]:
# Load loss criterion
criterion = get_loss(config["loss_name"], config["loss_kwargs"], config["model_kwargs"])
print(f"Loss criterion: {config['loss_name']}")
print(f"Loss weights: {config['loss_kwargs']}")

Loss criterion: set
Loss weights: {'loss_ce': 0.4, 'loss_mask': 400, 'loss_dice': 2, 'eos_coef': 1, 'clf_loss': 1.0}


## Interactive Sample Analysis Configuration

Configure which samples you want to analyze individually.

In [7]:
# Configuration for analysis
analysis_config = {
    "split_to_analyze": "test",  # "train", "valid", or "test"
}

print("Analysis Configuration:")
for key, value in analysis_config.items():
    print(f"  {key}: {value}")

print(f"\nDataset splits available:")
print(f"  Train: {len(dataset.idx_split['train'])} samples")
print(f"  Valid: {len(dataset.idx_split['valid'])} samples")
print(f"  Test: {len(dataset.idx_split['test'])} samples")

# Create single sample loader for individual analysis
def get_single_sample(split, idx):
    """Get a single sample from the dataset"""
    from torch_geometric.loader import DataLoader
    sample_indices = [dataset.idx_split[split][idx]]
    single_loader = DataLoader(
        dataset[sample_indices],
        batch_size=1,
        shuffle=False,
        num_workers=1
    )
    return next(iter(single_loader))

Analysis Configuration:
  split_to_analyze: test

Dataset splits available:
  Train: 1520 samples
  Valid: 190 samples
  Test: 190 samples


## Analysis Functions

In [8]:
def weighted_loss(losses, criterion):
    """Apply loss weights and return total loss"""
    for k in list(losses.keys()):
        if k in criterion.weight_dict:
            losses[k] *= criterion.weight_dict[k]
        else:
            print(f"Warning: {k} not in weight_dict, removing from losses")
            losses.pop(k)
    all_losses = sum(losses.values())
    all_losses_item_dict = {k: v.item() for k, v in losses.items()}
    return all_losses, all_losses_item_dict

def calc_clf_metrics(pred, data, criterion):
    """Calculate classification metrics"""
    if "clf_logits" not in pred:
        return None, None, None, None

    y_pred = pred["clf_logits"].squeeze(-1)
    y_true = ((data.pt > 0.9) & (data.reconstructable == 1)).float()
    loss = F.binary_cross_entropy_with_logits(y_pred, y_true)
    auroc = roc_auc_score(y_true.detach().cpu().numpy(), y_pred.detach().sigmoid().cpu().numpy())
    acc = ((y_pred.sigmoid() > 0.5) == y_true).float().mean().item()

    return loss * criterion.weight_dict.get("clf_loss", 1.0), auroc, acc, (y_pred.sigmoid().cpu().numpy(), y_true.cpu().numpy())

def calc_tracking_metrics(pred_masks, data):
    """Calculate tracking metrics (double majority)"""
    predicted = pred_masks[0, ..., 0].argmax(dim=0)
    truth = data.targets[0][0]["masks"][..., 0].argmax(dim=0)
    pts = data.pt
    reconstructable = data.reconstructable

    res = tracking_metrics(
        truth=truth.cpu().numpy(),
        predicted=predicted.cpu().numpy(),
        pts=pts.cpu().numpy(),
        reconstructable=reconstructable.cpu().numpy(),
        pt_thlds=[0.9],
        predicted_count_thld=3,
    )
    return res[0.9]["double_majority"], predicted.cpu().numpy(), truth.cpu().numpy()

def analyze_batch(model, criterion, data):
    """Analyze a single batch and return detailed metrics"""
    with torch.no_grad():
        pred = model(data)

        # Calculate losses
        loss = criterion(pred, data.targets[0])
        total_loss, loss_dict = weighted_loss(loss, criterion)

        # Calculate classification metrics
        clf_loss, clf_auroc, clf_acc, clf_preds = calc_clf_metrics(pred, data, criterion)
        if clf_loss is not None:
            total_loss += clf_loss
            loss_dict["clf_loss"] = clf_loss.item()
            loss_dict["clf_auroc"] = clf_auroc
            loss_dict["clf_acc"] = clf_acc

        # Calculate tracking metrics
        pred_masks = pred["pred_masks"].detach()
        dm_score, predicted_tracks, truth_tracks = calc_tracking_metrics(pred_masks, data)

        loss_dict["total_loss"] = total_loss.item()
        loss_dict["dm_score"] = dm_score

        return {
            "losses": loss_dict,
            "predictions": {
                "pred_masks": pred_masks.cpu().numpy(),
                "pred_logits": pred["pred_logits"].cpu().numpy() if "pred_logits" in pred else None,
                "clf_preds": clf_preds,
                "predicted_tracks": predicted_tracks,
                "truth_tracks": truth_tracks
            },
        }

## Run Analysis on Test Set

In [20]:
# Use configuration for batch analysis
split_to_analyze = analysis_config["split_to_analyze"]

results = []
data_loader = loaders[split_to_analyze]

for idx, data in enumerate(tqdm(data_loader, desc=f"Analyzing {split_to_analyze} samples")):

    data = data.to(device)
    result = analyze_batch(model, criterion, data)
    result["sample_idx"] = idx
    results.append(result)
    if result["losses"]["dm_score"] < 0.85:
        print(f"Sample {idx} has low DM score: {result['losses']['dm_score']:.4f}")

print(f"Batch analysis completed for {len(results)} samples!")

# Quick summary
dm_scores = [r["losses"]["dm_score"] for r in results]
total_losses = [r["losses"]["total_loss"] for r in results]
print(f"Quick Summary:")
print(f"  Mean DM Score: {np.mean(dm_scores):.4f} ± {np.std(dm_scores):.4f}")
print(f"  Mean Total Loss: {np.mean(total_losses):.4f} ± {np.std(total_losses):.4f}")
print(f"  Best DM Score: {np.max(dm_scores):.4f} (sample {np.argmax(dm_scores)})")
print(f"  Worst DM Score: {np.min(dm_scores):.4f} (sample {np.argmin(dm_scores)})")

Analyzing test samples:   1%|          | 2/190 [00:14<19:58,  6.38s/it]

Sample 1 has low DM score: 0.8344


Analyzing test samples:   3%|▎         | 5/190 [00:18<07:10,  2.33s/it]

Sample 3 has low DM score: 0.8256
Sample 4 has low DM score: 0.8465


Analyzing test samples:   3%|▎         | 6/190 [00:18<04:51,  1.58s/it]

...

Sample 183 has low DM score: 0.7849


Analyzing test samples:  99%|█████████▉| 188/190 [01:00<00:00,  6.26it/s]

Sample 186 has low DM score: 0.8367


Analyzing test samples: 100%|██████████| 190/190 [01:01<00:00,  3.10it/s]

Sample 188 has low DM score: 0.8449
Batch analysis completed for 190 samples!
Quick Summary:
  Mean DM Score: 0.8692 ± 0.0380
  Mean Total Loss: 1.3604 ± 0.2233
  Best DM Score: 0.9390 (sample 86)
  Worst DM Score: 0.7202 (sample 83)


In [9]:
# === ANALYZE INDIVIDUAL SAMPLE ===
# Change this index to analyze different samples (0, 1, 2, ...)
sample_idx = 90  # Change this to analyze different samples
split = "test"   # Change to "train", "valid", or "test"

print(f"Analyzing sample {sample_idx} from {split} set...")

# Get single sample
data = get_single_sample(split, sample_idx).to(device)
result = analyze_batch(model, criterion, data)
print(result['losses']['dm_score'])

Analyzing sample 90 from test set...
0.7972805933250927


In [10]:
pred_masks = result['predictions']["pred_masks"][0, ..., 0]
gt_masks = data.targets[0][0]["masks"][..., 0]

In [11]:
gt_masks.shape, pred_masks.shape

(torch.Size([1618, 10990]), (1950, 10990))

In [25]:
pred_cluster = torch.tensor(pred_masks.argmax(0))
true_cluster = gt_masks.argmax(0)
pred_cluster.shape, true_cluster.shape

(torch.Size([10990]), torch.Size([10990]))

In [75]:
_, cnt = torch.unique(pred_cluster, return_counts=True)
_, cnt_true = torch.unique(true_cluster, return_counts=True)

In [ ]:
(cnt == 1).sum(), (cnt_true == 2).sum()

(tensor(88), tensor(0, device='cuda:5'))

: 

In [29]:
idx = 0

In [30]:
torch.where(pred_cluster == pred_cluster[idx]), torch.where(true_cluster == true_cluster[idx])

((tensor([   0,    3,    8,   10,  244,  245,  252,  253,  534,  554,  895,  908,
          1329, 1339]),),
 (tensor([   0,   10,  245,  252,  534,  554,  895,  908, 1329, 1339],
         device='cuda:5'),))

In [57]:
torch.where(true_cluster == true_cluster[3])

(tensor([   3,    8,  244,  253,  533,  552,  899,  910, 1323, 1344],
        device='cuda:5'),)

In [62]:
pred_cluster[244]

tensor(1155)

In [56]:
torch.where(pred_cluster == pred_cluster[533])

(tensor([   2,  243,  532,  533,  545,  552,  893,  899,  910,  912, 1323, 1330,
         1336, 1344, 1829, 1842, 2316, 2322, 2334, 2810]),)

In [35]:
torch.where(true_cluster == true_cluster[2])

(tensor([   2,  243,  532,  545,  893,  912, 1330, 1336, 1829, 1842, 2316, 2322,
         2334, 2810], device='cuda:5'),)

In [43]:
pred_masks[:, 3].sort()

In [68]:
torch.tensor(pred_masks[:, 253]).argsort()

tensor([ 513,  400, 1598,  ...,  669, 1770, 1155])

In [69]:
torch.where(pred_cluster == 1770)

(tensor([   2,  243,  532,  533,  545,  552,  893,  899,  910,  912, 1323, 1330,
         1336, 1344, 1829, 1842, 2316, 2322, 2334, 2810]),)

In [27]:
result

{'losses': {'loss_ce': 0.309948205947876,
  'loss_mask': 0.1959078013896942,
  'loss_dice': 0.4045127332210541,
  'loss_ce_0': 0.30997928977012634,
  'loss_mask_0': 0.18315716087818146,
  'loss_dice_0': 0.39931514859199524,
  'total_loss': 1.8028204441070557,
  'dm_score': 0.7972805933250927},
 'predictions': {'pred_masks': array([[[[-1370.1877 ],
           [ -145.99963],
           [-1342.4794 ],
           ...,
           [-1559.2595 ],
           [-1475.1118 ],
           [-1356.0981 ]],
  
          [[ -671.3251 ],
           [ -114.62046],
           [ -801.06744],
           ...,
           [-1176.999  ],
           [ -945.3586 ],
           [-1054.4958 ]],
  
          [[-1101.0977 ],
           [ -104.05534],
           [ -838.2542 ],
           ...,
           [-1241.6439 ],
           [-1207.4446 ],
           [ -895.68085]],
  
          ...,
  
          [[ -390.41678],
           [  -65.99453],
           [ -382.17102],
           ...,
           [ -694.02783],
          

## Individual Sample Analysis

Analyze a single sample in detail. Change the `sample_idx` below to analyze different samples.

In [ ]:
# === ANALYZE INDIVIDUAL SAMPLE ===
# Change this index to analyze different samples (0, 1, 2, ...)
sample_idx = 83  # Change this to analyze different samples
split = "test"   # Change to "train", "valid", or "test"

print(f"Analyzing sample {sample_idx} from {split} set...")

# Get single sample
data = get_single_sample(split, sample_idx).to(device)
result = analyze_batch(model, criterion, data)

print(f"\n=== SAMPLE {sample_idx} ANALYSIS ===")
print(f"Dataset split: {split}")
print(f"Number of hits: {result['data_info']['num_hits']}")
print(f"Number of true tracks: {result['data_info']['num_true_tracks']}")
print(f"DM Score: {result['losses']['dm_score']:.4f}")
print(f"Total Loss: {result['losses']['total_loss']:.4f}")

print(f"\nLoss breakdown:")
for loss_name, loss_value in result['losses'].items():
    if loss_name not in ['dm_score', 'total_loss']:
        print(f"  {loss_name}: {loss_value:.6f}")

# Get predictions and ground truth
pred_masks = result['predictions']['pred_masks']
predicted_tracks = result['predictions']['predicted_tracks']
truth_tracks = result['predictions']['truth_tracks']
pt_values = result['data_info']['pt']
reconstructable = result['data_info']['reconstructable']

print(f"\nTrack assignment summary:")
print(f"  Unique predicted tracks: {len(np.unique(predicted_tracks))}")
print(f"  Unique true tracks: {len(np.unique(truth_tracks))}")
print(f"  Hits with pt > 0.9: {np.sum(pt_values > 0.9)}")
print(f"  Reconstructable hits: {np.sum(reconstructable)}")

# Visualize track assignments
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# True track assignments
scatter1 = axes[0, 0].scatter(range(len(truth_tracks)), truth_tracks,
                             c=truth_tracks, cmap='tab20', alpha=0.7, s=30)
axes[0, 0].set_title(f'True Track Assignments (Sample {sample_idx})')
axes[0, 0].set_xlabel('Hit Index')
axes[0, 0].set_ylabel('Track ID')
axes[0, 0].grid(True, alpha=0.3)

# Predicted track assignments
scatter2 = axes[0, 1].scatter(range(len(predicted_tracks)), predicted_tracks,
                             c=predicted_tracks, cmap='tab20', alpha=0.7, s=30)
axes[0, 1].set_title(f'Predicted Track Assignments (Sample {sample_idx})')
axes[0, 1].set_xlabel('Hit Index')
axes[0, 1].set_ylabel('Track ID')
axes[0, 1].grid(True, alpha=0.3)

# PT distribution colored by track assignment
axes[1, 0].scatter(range(len(pt_values)), pt_values,
                  c=truth_tracks, cmap='tab20', alpha=0.7, s=30)
axes[1, 0].axhline(y=0.9, color='red', linestyle='--', alpha=0.7, label='PT threshold')
axes[1, 0].set_title('PT Values (colored by true tracks)')
axes[1, 0].set_xlabel('Hit Index')
axes[1, 0].set_ylabel('PT')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Reconstructable hits
recon_colors = ['red' if r else 'blue' for r in reconstructable]
axes[1, 1].scatter(range(len(reconstructable)), reconstructable,
                  c=recon_colors, alpha=0.7, s=30)
axes[1, 1].set_title('Reconstructable Hits')
axes[1, 1].set_xlabel('Hit Index')
axes[1, 1].set_ylabel('Reconstructable (1=Yes, 0=No)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n=== READY FOR NEXT SAMPLE ===")
print(f"To analyze another sample, change 'sample_idx' above and re-run this cell.")
print(f"Available indices for {split} set: 0 to {len(dataset.idx_split[split])-1}")

Analyzing sample 83 from test set...


## Compare Multiple Samples

Compare several samples side by side to understand patterns.

In [ ]:
# === COMPARE MULTIPLE SAMPLES ===
# Specify which samples to compare (indices from the split)
samples_to_compare = [0, 1, 2]  # Change these indices
comparison_split = "test"  # Change to analyze different split

print(f"Comparing samples {samples_to_compare} from {comparison_split} set...")

comparison_results = []
for sample_idx in samples_to_compare:
    data = get_single_sample(comparison_split, sample_idx).to(device)
    result = analyze_batch(model, criterion, data)
    result["sample_idx"] = sample_idx
    comparison_results.append(result)

    print(f"Sample {sample_idx}: DM={result['losses']['dm_score']:.4f}, "
          f"Loss={result['losses']['total_loss']:.4f}, "
          f"Hits={result['data_info']['num_hits']}, "
          f"Tracks={result['data_info']['num_true_tracks']}")

# Create comparison visualization
fig, axes = plt.subplots(len(samples_to_compare), 3, figsize=(18, 6*len(samples_to_compare)))
if len(samples_to_compare) == 1:
    axes = axes.reshape(1, -1)

for i, (sample_idx, result) in enumerate(zip(samples_to_compare, comparison_results)):
    predicted_tracks = result['predictions']['predicted_tracks']
    truth_tracks = result['predictions']['truth_tracks']
    pt_values = result['data_info']['pt']

    # True tracks
    axes[i, 0].scatter(range(len(truth_tracks)), truth_tracks,
                      c=truth_tracks, cmap='tab20', alpha=0.7, s=20)
    axes[i, 0].set_title(f'Sample {sample_idx}: True Tracks (DM={result["losses"]["dm_score"]:.3f})')
    axes[i, 0].set_xlabel('Hit Index')
    axes[i, 0].set_ylabel('Track ID')
    axes[i, 0].grid(True, alpha=0.3)

    # Predicted tracks
    axes[i, 1].scatter(range(len(predicted_tracks)), predicted_tracks,
                      c=predicted_tracks, cmap='tab20', alpha=0.7, s=20)
    axes[i, 1].set_title(f'Sample {sample_idx}: Predicted Tracks')
    axes[i, 1].set_xlabel('Hit Index')
    axes[i, 1].set_ylabel('Track ID')
    axes[i, 1].grid(True, alpha=0.3)

    # PT distribution
    axes[i, 2].scatter(range(len(pt_values)), pt_values,
                      c=truth_tracks, cmap='tab20', alpha=0.7, s=20)
    axes[i, 2].axhline(y=0.9, color='red', linestyle='--', alpha=0.7)
    axes[i, 2].set_title(f'Sample {sample_idx}: PT Values')
    axes[i, 2].set_xlabel('Hit Index')
    axes[i, 2].set_ylabel('PT')
    axes[i, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print detailed comparison
print(f"\n=== DETAILED COMPARISON ===")
for i, (sample_idx, result) in enumerate(zip(samples_to_compare, comparison_results)):
    print(f"\nSample {sample_idx}:")
    print(f"  DM Score: {result['losses']['dm_score']:.4f}")
    print(f"  Total Loss: {result['losses']['total_loss']:.4f}")
    print(f"  Hits: {result['data_info']['num_hits']}")
    print(f"  True Tracks: {result['data_info']['num_true_tracks']}")
    print(f"  Loss breakdown:")
    for loss_name, loss_value in result['losses'].items():
        if loss_name not in ['dm_score', 'total_loss']:
            print(f"    {loss_name}: {loss_value:.6f}")

print(f"\nTo compare different samples, modify 'samples_to_compare' above and re-run.")

## Loss Analysis

In [ ]:
# Extract loss statistics
loss_data = pd.DataFrame([r["losses"] for r in results])
loss_data["sample_idx"] = range(len(results))

print("Loss Statistics:")
print(loss_data.describe())

# Plot loss distributions
loss_columns = [col for col in loss_data.columns if col != "sample_idx"]
n_cols = min(3, len(loss_columns))
n_rows = (len(loss_columns) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
if n_rows == 1:
    axes = [axes] if n_cols == 1 else axes
else:
    axes = axes.flatten()

for i, col in enumerate(loss_columns):
    if i < len(axes):
        axes[i].hist(loss_data[col], bins=20, alpha=0.7, edgecolor='black')
        axes[i].set_title(f'{col} Distribution')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')
        axes[i].grid(True, alpha=0.3)

# Hide unused subplots
for i in range(len(loss_columns), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Plot loss correlations
plt.figure(figsize=(10, 8))
correlation_matrix = loss_data[loss_columns].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Loss Correlation Matrix')
plt.tight_layout()
plt.show()

print("\nTop Loss Correlations:")
corr_pairs = []
for i in range(len(loss_columns)):
    for j in range(i+1, len(loss_columns)):
        corr_pairs.append((loss_columns[i], loss_columns[j], correlation_matrix.iloc[i, j]))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for col1, col2, corr in corr_pairs[:5]:
    print(f"{col1} vs {col2}: {corr:.3f}")

## Tracking Performance Analysis

In [ ]:
# Extract tracking metrics
dm_scores = [r["losses"]["dm_score"] for r in results]
num_hits = [r["data_info"]["num_hits"] for r in results]
num_true_tracks = [r["data_info"]["num_true_tracks"] for r in results]

print(f"Double Majority Score Statistics:")
print(f"Mean: {np.mean(dm_scores):.4f}")
print(f"Std: {np.std(dm_scores):.4f}")
print(f"Min: {np.min(dm_scores):.4f}")
print(f"Max: {np.max(dm_scores):.4f}")

# Plot DM score distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# DM score histogram
axes[0, 0].hist(dm_scores, bins=20, alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Double Majority Score Distribution')
axes[0, 0].set_xlabel('DM Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

# DM score vs number of hits
axes[0, 1].scatter(num_hits, dm_scores, alpha=0.7)
axes[0, 1].set_title('DM Score vs Number of Hits')
axes[0, 1].set_xlabel('Number of Hits')
axes[0, 1].set_ylabel('DM Score')
axes[0, 1].grid(True, alpha=0.3)

# DM score vs number of true tracks
axes[1, 0].scatter(num_true_tracks, dm_scores, alpha=0.7)
axes[1, 0].set_title('DM Score vs Number of True Tracks')
axes[1, 0].set_xlabel('Number of True Tracks')
axes[1, 0].set_ylabel('DM Score')
axes[1, 0].grid(True, alpha=0.3)

# DM score vs total loss
total_losses = [r["losses"]["total_loss"] for r in results]
axes[1, 1].scatter(total_losses, dm_scores, alpha=0.7)
axes[1, 1].set_title('DM Score vs Total Loss')
axes[1, 1].set_xlabel('Total Loss')
axes[1, 1].set_ylabel('DM Score')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Classification Analysis (if available)

In [ ]:
# Check if classification predictions are available
has_clf = any(r["predictions"]["clf_preds"] is not None for r in results)

if has_clf:
    print("Classification Analysis:")

    # Collect all classification predictions and targets
    all_clf_preds = []
    all_clf_targets = []
    clf_aucs = []
    clf_accs = []

    for r in results:
        if r["predictions"]["clf_preds"] is not None:
            clf_pred, clf_target = r["predictions"]["clf_preds"]
            all_clf_preds.extend(clf_pred)
            all_clf_targets.extend(clf_target)
            clf_aucs.append(r["losses"]["clf_auroc"])
            clf_accs.append(r["losses"]["clf_acc"])

    all_clf_preds = np.array(all_clf_preds)
    all_clf_targets = np.array(all_clf_targets)

    print(f"Classification AUC per sample - Mean: {np.mean(clf_aucs):.4f}, Std: {np.std(clf_aucs):.4f}")
    print(f"Classification Accuracy per sample - Mean: {np.mean(clf_accs):.4f}, Std: {np.std(clf_accs):.4f}")

    # Overall classification metrics
    overall_auc = roc_auc_score(all_clf_targets, all_clf_preds)
    overall_acc = ((all_clf_preds > 0.5) == all_clf_targets).mean()

    print(f"Overall AUC: {overall_auc:.4f}")
    print(f"Overall Accuracy: {overall_acc:.4f}")

    # Plot classification analysis
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Prediction distribution by class
    axes[0, 0].hist(all_clf_preds[all_clf_targets == 0], bins=50, alpha=0.7, label='Negative', density=True)
    axes[0, 0].hist(all_clf_preds[all_clf_targets == 1], bins=50, alpha=0.7, label='Positive', density=True)
    axes[0, 0].set_title('Prediction Distribution by Class')
    axes[0, 0].set_xlabel('Prediction Probability')
    axes[0, 0].set_ylabel('Density')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # ROC Curve
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(all_clf_targets, all_clf_preds)
    axes[0, 1].plot(fpr, tpr, label=f'ROC (AUC = {overall_auc:.3f})')
    axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[0, 1].set_title('ROC Curve')
    axes[0, 1].set_xlabel('False Positive Rate')
    axes[0, 1].set_ylabel('True Positive Rate')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Confusion Matrix
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(all_clf_targets, all_clf_preds > 0.5)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0])
    axes[1, 0].set_title('Confusion Matrix')
    axes[1, 0].set_xlabel('Predicted')
    axes[1, 0].set_ylabel('Actual')

    # AUC distribution across samples
    axes[1, 1].hist(clf_aucs, bins=20, alpha=0.7, edgecolor='black')
    axes[1, 1].set_title('AUC Distribution Across Samples')
    axes[1, 1].set_xlabel('AUC')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No classification predictions available in the model output.")

## Error Pattern Analysis

In [ ]:
# Identify worst performing samples
performance_df = pd.DataFrame({
    'sample_idx': range(len(results)),
    'dm_score': dm_scores,
    'total_loss': total_losses,
    'num_hits': num_hits,
    'num_true_tracks': num_true_tracks
})

# Add loss components
for col in loss_columns:
    if col not in ['dm_score', 'total_loss']:
        performance_df[col] = loss_data[col]

# Sort by performance
worst_samples = performance_df.nsmallest(5, 'dm_score')
best_samples = performance_df.nlargest(5, 'dm_score')

print("5 Worst Performing Samples (lowest DM score):")
print(worst_samples)
print("\n5 Best Performing Samples (highest DM score):")
print(best_samples)

# Analyze patterns
print("\nPattern Analysis:")
print(f"Correlation between DM score and total loss: {np.corrcoef(dm_scores, total_losses)[0,1]:.3f}")
print(f"Correlation between DM score and number of hits: {np.corrcoef(dm_scores, num_hits)[0,1]:.3f}")
print(f"Correlation between DM score and number of true tracks: {np.corrcoef(dm_scores, num_true_tracks)[0,1]:.3f}")

## Detailed Sample Analysis

In [ ]:
# Analyze a specific sample in detail
sample_to_analyze = worst_samples.iloc[0]['sample_idx']  # Analyze the worst sample
sample_result = results[int(sample_to_analyze)]

print(f"Detailed Analysis for Sample {sample_to_analyze} (worst performing):")
print(f"DM Score: {sample_result['losses']['dm_score']:.4f}")
print(f"Total Loss: {sample_result['losses']['total_loss']:.4f}")
print(f"Number of Hits: {sample_result['data_info']['num_hits']}")
print(f"Number of True Tracks: {sample_result['data_info']['num_true_tracks']}")

print("\nLoss Breakdown:")
for loss_name, loss_value in sample_result['losses'].items():
    if loss_name not in ['dm_score', 'total_loss']:
        print(f"{loss_name}: {loss_value:.4f}")

# Visualize predicted vs true tracks
predicted_tracks = sample_result['predictions']['predicted_tracks']
truth_tracks = sample_result['predictions']['truth_tracks']

print(f"\nTrack Statistics:")
print(f"Unique predicted tracks: {len(np.unique(predicted_tracks))}")
print(f"Unique true tracks: {len(np.unique(truth_tracks))}")

# Plot track assignment comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# True tracks
axes[0].scatter(range(len(truth_tracks)), truth_tracks, alpha=0.7, s=20)
axes[0].set_title('True Track Assignments')
axes[0].set_xlabel('Hit Index')
axes[0].set_ylabel('Track ID')
axes[0].grid(True, alpha=0.3)

# Predicted tracks
axes[1].scatter(range(len(predicted_tracks)), predicted_tracks, alpha=0.7, s=20, color='orange')
axes[1].set_title('Predicted Track Assignments')
axes[1].set_xlabel('Hit Index')
axes[1].set_ylabel('Track ID')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary and Recommendations

In [ ]:
print("=" * 60)
print("SUMMARY AND RECOMMENDATIONS")
print("=" * 60)

print(f"\n1. OVERALL PERFORMANCE:")
print(f"   - Mean DM Score: {np.mean(dm_scores):.4f} ± {np.std(dm_scores):.4f}")
print(f"   - Mean Total Loss: {np.mean(total_losses):.4f} ± {np.std(total_losses):.4f}")

print(f"\n2. LOSS COMPONENT ANALYSIS:")
for col in loss_columns:
    if col not in ['dm_score', 'total_loss']:
        mean_val = loss_data[col].mean()
        std_val = loss_data[col].std()
        print(f"   - {col}: {mean_val:.4f} ± {std_val:.4f}")

print(f"\n3. CORRELATIONS:")
print(f"   - DM Score vs Total Loss: {np.corrcoef(dm_scores, total_losses)[0,1]:.3f}")
print(f"   - DM Score vs Number of Hits: {np.corrcoef(dm_scores, num_hits)[0,1]:.3f}")
print(f"   - DM Score vs Number of True Tracks: {np.corrcoef(dm_scores, num_true_tracks)[0,1]:.3f}")

if has_clf:
    print(f"\n4. CLASSIFICATION PERFORMANCE:")
    print(f"   - Overall AUC: {overall_auc:.4f}")
    print(f"   - Overall Accuracy: {overall_acc:.4f}")

print(f"\n5. RECOMMENDATIONS FOR IMPROVEMENT:")

# Analyze which loss component contributes most
loss_means = {col: loss_data[col].mean() for col in loss_columns if col not in ['dm_score', 'total_loss']}
dominant_loss = max(loss_means, key=loss_means.get)
print(f"   - Focus on {dominant_loss} (highest average loss component)")

# Check if there's high variance in performance
dm_cv = np.std(dm_scores) / np.mean(dm_scores)
if dm_cv > 0.3:
    print(f"   - High performance variance detected (CV={dm_cv:.3f}). Consider:")
    print(f"     * Data augmentation or regularization")
    print(f"     * Analysis of outlier samples")

# Check correlation patterns
if abs(np.corrcoef(dm_scores, num_hits)[0,1]) > 0.3:
    print(f"   - Strong correlation between performance and number of hits")
    print(f"     * Consider hit-adaptive models or preprocessing")

if abs(np.corrcoef(dm_scores, total_losses)[0,1]) < -0.5:
    print(f"   - Good alignment between loss and performance metric")
else:
    print(f"   - Weak alignment between loss and performance metric")
    print(f"     * Consider adjusting loss weights or adding task-specific losses")

print(f"\n6. NEXT STEPS:")
print(f"   - Examine worst-performing samples for common patterns")
print(f"   - Consider ablation studies on loss components")
print(f"   - Analyze model attention/feature importance if available")
print(f"   - Compare with baseline models or previous checkpoints")

print("\n" + "=" * 60)

## Save Results for Further Analysis

In [ ]:
# Save detailed results
import pickle

# Create output directory
output_dir = Path("./analysis_results")
output_dir.mkdir(exist_ok=True)

# Save performance dataframe
performance_df.to_csv(output_dir / "performance_summary.csv", index=False)
print(f"Performance summary saved to {output_dir / 'performance_summary.csv'}")

# Save detailed results (predictions, etc.)
with open(output_dir / "detailed_results.pkl", "wb") as f:
    pickle.dump(results, f)
print(f"Detailed results saved to {output_dir / 'detailed_results.pkl'}")

# Save analysis summary
summary = {
    "config": config,
    "num_samples_analyzed": len(results),
    "mean_dm_score": np.mean(dm_scores),
    "std_dm_score": np.std(dm_scores),
    "mean_total_loss": np.mean(total_losses),
    "loss_statistics": loss_data.describe().to_dict(),
    "worst_samples": worst_samples.to_dict(),
    "best_samples": best_samples.to_dict()
}

with open(output_dir / "analysis_summary.pkl", "wb") as f:
    pickle.dump(summary, f)
print(f"Analysis summary saved to {output_dir / 'analysis_summary.pkl'}")

print("\nAnalysis complete! Check the analysis_results directory for saved outputs.")

## Advanced Analysis Tools

Additional utilities for deeper analysis.

In [ ]:
# === ADVANCED ANALYSIS UTILITIES ===

def analyze_prediction_masks(pred_masks, truth_masks, sample_name=""):
    """Analyze prediction mask details"""
    print(f"=== Prediction Mask Analysis {sample_name} ===")
    print(f"Prediction masks shape: {pred_masks.shape}")
    print(f"Truth masks shape: {truth_masks.shape}")

    # Mask statistics
    pred_max_vals = np.max(pred_masks, axis=(0, 2))  # Max over batch and last dim
    pred_min_vals = np.min(pred_masks, axis=(0, 2))

    print(f"Prediction mask statistics:")
    print(f"  Max values per query: {pred_max_vals[:10]}...")  # Show first 10
    print(f"  Min values per query: {pred_min_vals[:10]}...")
    print(f"  Overall max: {np.max(pred_masks):.4f}")
    print(f"  Overall min: {np.min(pred_masks):.4f}")

    # Plot mask activation patterns
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Heatmap of prediction masks (first few queries)
    n_queries_to_show = min(20, pred_masks.shape[1])
    axes[0].imshow(pred_masks[0, :n_queries_to_show, :, 0].T, aspect='auto', cmap='viridis')
    axes[0].set_title(f'Prediction Masks Heatmap {sample_name}')
    axes[0].set_xlabel('Query Index')
    axes[0].set_ylabel('Hit Index')

    # Distribution of max activations per query
    max_activations = np.max(pred_masks[0, ..., 0], axis=1)
    axes[1].hist(max_activations, bins=30, alpha=0.7, edgecolor='black')
    axes[1].set_title('Distribution of Max Activations per Query')
    axes[1].set_xlabel('Max Activation Value')
    axes[1].set_ylabel('Frequency')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def find_similar_samples(target_idx, results, metric='dm_score', n_similar=3):
    """Find samples with similar performance"""
    target_score = results[target_idx]["losses"][metric]

    # Calculate distances
    distances = []
    for i, result in enumerate(results):
        if i != target_idx:
            score = result["losses"][metric]
            distance = abs(score - target_score)
            distances.append((i, score, distance))

    # Sort by distance
    distances.sort(key=lambda x: x[2])

    print(f"Samples most similar to sample {target_idx} (target {metric}={target_score:.4f}):")
    for i in range(min(n_similar, len(distances))):
        idx, score, distance = distances[i]
        print(f"  Sample {idx}: {metric}={score:.4f} (distance={distance:.4f})")

    return [d[0] for d in distances[:n_similar]]

def analyze_failure_modes(results, dm_threshold=0.3):
    """Analyze common failure patterns"""
    print(f"=== FAILURE MODE ANALYSIS ===")

    # Identify failed samples
    failed_samples = [i for i, r in enumerate(results) if r["losses"]["dm_score"] < dm_threshold]
    success_samples = [i for i, r in enumerate(results) if r["losses"]["dm_score"] >= dm_threshold]

    print(f"Failed samples (DM < {dm_threshold}): {len(failed_samples)}")
    print(f"Successful samples (DM >= {dm_threshold}): {len(success_samples)}")

    if len(failed_samples) == 0:
        print("No failed samples found!")
        return

    # Compare characteristics
    failed_hits = [results[i]["data_info"]["num_hits"] for i in failed_samples]
    success_hits = [results[i]["data_info"]["num_hits"] for i in success_samples]

    failed_tracks = [results[i]["data_info"]["num_true_tracks"] for i in failed_samples]
    success_tracks = [results[i]["data_info"]["num_true_tracks"] for i in success_samples]

    print(f"\nCharacteristics comparison:")
    print(f"  Failed samples - Hits: {np.mean(failed_hits):.1f}±{np.std(failed_hits):.1f}")
    print(f"  Success samples - Hits: {np.mean(success_hits):.1f}±{np.std(success_hits):.1f}")
    print(f"  Failed samples - Tracks: {np.mean(failed_tracks):.1f}±{np.std(failed_tracks):.1f}")
    print(f"  Success samples - Tracks: {np.mean(success_tracks):.1f}±{np.std(success_tracks):.1f}")

    # Visualize differences
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].hist(failed_hits, bins=15, alpha=0.7, label='Failed', color='red')
    axes[0].hist(success_hits, bins=15, alpha=0.7, label='Success', color='green')
    axes[0].set_title('Number of Hits Distribution')
    axes[0].set_xlabel('Number of Hits')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].hist(failed_tracks, bins=15, alpha=0.7, label='Failed', color='red')
    axes[1].hist(success_tracks, bins=15, alpha=0.7, label='Success', color='green')
    axes[1].set_title('Number of True Tracks Distribution')
    axes[1].set_xlabel('Number of True Tracks')
    axes[1].set_ylabel('Frequency')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return failed_samples, success_samples

# Example usage (uncomment to use after running batch analysis):
# if 'results' in locals() and len(results) > 0:
#     # Analyze a specific sample's prediction masks
#     sample_to_analyze = 0
#     analyze_prediction_masks(
#         results[sample_to_analyze]["predictions"]["pred_masks"],
#         results[sample_to_analyze]["predictions"]["truth_tracks"],
#         f"(Sample {sample_to_analyze})"
#     )
#
#     # Find similar samples
#     similar_samples = find_similar_samples(0, results)
#
#     # Analyze failure modes
#     failed, success = analyze_failure_modes(results)

print("Advanced analysis tools loaded!")
print("Functions available:")
print("  - analyze_prediction_masks(pred_masks, truth_masks, sample_name)")
print("  - find_similar_samples(target_idx, results, metric, n_similar)")
print("  - analyze_failure_modes(results, dm_threshold)")
print("\nUncomment the example usage section to run analyses automatically.")